In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Allowed components list
allowed_cmdb = [
    "apache01", "apache02", "Tomcat01", "Tomcat02", "Tomcat04", "Tomcat03",
    "MG01", "MG02", "IG01", "IG02", "Mysql01", "Mysql02", "Redis01", "Redis02"
]

# Timezone setup (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (full DataFrames kept for later steps)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Filter to allowed cmdb_id values
metric_df = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()
trace_df = trace_df[trace_df['cmdb_id'].isin(allowed_cmdb)].copy()
log_df = log_df[log_df['cmdb_id'].isin(allowed_cmdb)].copy()
error_logs_df = error_logs_df[error_logs_df['cmdb_id'].isin(allowed_cmdb)].copy()

# Helper to aggregate a dataframe with (cmdb_id, kpi_name) grouping
def aggregate_kpi(df, kpi_col, file_type_label):
    # Ensure value and timestamp columns have correct types
    df = df.copy()
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    # Drop rows with NaN timestamp or value (can't compute percentile otherwise)
    df = df.dropna(subset=['timestamp'])
    # unify kpi column name
    df = df.rename(columns={kpi_col: 'kpi_name'})[['cmdb_id', 'kpi_name', 'timestamp', 'value']]
    # Group and compute aggregations
    grouped = df.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
        count_points = ('value', 'count'),
        min_ts = ('timestamp', 'min'),
        max_ts = ('timestamp', 'max'),
        p95_value = ('value', lambda s: float(np.nan) if s.dropna().size==0 else float(s.quantile(0.95)))
    ).reset_index()
    # Convert timestamps to timezone-aware datetimes in Asia/Shanghai
    grouped['min_timestamp'] = pd.to_datetime(grouped['min_ts'].astype('float'), unit='s', utc=True).dt.tz_convert(tz_sh)
    grouped['max_timestamp'] = pd.to_datetime(grouped['max_ts'].astype('float'), unit='s', utc=True).dt.tz_convert(tz_sh)
    # Keep required columns and add file_type
    grouped['file_type'] = file_type_label
    result = grouped[['file_type', 'cmdb_id', 'kpi_name', 'count_points', 'min_timestamp', 'max_timestamp', 'p95_value']]
    # sort by count desc for convenience
    result = result.sort_values(by='count_points', ascending=False).reset_index(drop=True)
    return result

# Aggregate metric, trace, log
metric_agg = aggregate_kpi(metric_df, 'kpi_name', 'metric')
trace_agg = aggregate_kpi(trace_df, 'trace_name', 'trace')
log_agg = aggregate_kpi(log_df, 'log_name', 'log')

# Combine up to top 50 groups across all files sorted by count desc
combined_agg = pd.concat([metric_agg, trace_agg, log_agg], ignore_index=True)
combined_agg = combined_agg.sort_values(by='count_points', ascending=False).reset_index(drop=True)
combined_display = combined_agg.head(50).copy()
# Rename count_points to count for output compactness and round p95
combined_display = combined_display.rename(columns={'count_points':'count'})
combined_display['p95_value'] = combined_display['p95_value'].round(6)

# Error logs summary: for allowed cmdb_id compute count and earliest/latest timestamp; only count>0 and top 20
if not error_logs_df.empty:
    error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce').astype('Int64')
    err_grp = error_logs_df.groupby('cmdb_id', dropna=False).agg(
        count = ('timestamp', 'count'),
        min_ts = ('timestamp', 'min'),
        max_ts = ('timestamp', 'max')
    ).reset_index()
    # keep only count>0 and convert timestamps to tz-aware datetimes
    err_grp = err_grp[err_grp['count'] > 0].copy()
    if not err_grp.empty:
        err_grp['earliest_timestamp'] = pd.to_datetime(err_grp['min_ts'].astype('float'), unit='s', utc=True).dt.tz_convert(tz_sh)
        err_grp['latest_timestamp'] = pd.to_datetime(err_grp['max_ts'].astype('float'), unit='s', utc=True).dt.tz_convert(tz_sh)
        error_summary = err_grp[['cmdb_id', 'count', 'earliest_timestamp', 'latest_timestamp']].sort_values(by='count', ascending=False).reset_index(drop=True)
        error_display = error_summary.head(20).copy()
    else:
        error_display = pd.DataFrame(columns=['cmdb_id', 'count', 'earliest_timestamp', 'latest_timestamp'])
else:
    error_display = pd.DataFrame(columns=['cmdb_id', 'count', 'earliest_timestamp', 'latest_timestamp'])

# Final compact outputs: combined_display (up to top 50 groups) and error_display (top 20 error log counts)
combined_display, error_display

```
Out[1]:
```
```python
# Summarize the previous aggregation results in plain English (compact)
# Reuse combined_display and error_display variables from prior execution

# Prepare top examples (up to 3) for inclusion in the summary
top_n = combined_display.head(3)
examples = []
for _, r in top_n.iterrows():
    examples.append(f"{r['file_type']} | {r['cmdb_id']} | {r['kpi_name']} | count={int(r['count'])} | p95={float(r['p95_value']):.6f}")

# Compute basic stats for the displayed aggregate table
displayed_groups = int(combined_display.shape[0])
max_count = int(combined_display['count'].max()) if displayed_groups>0 else 0
# determine global earliest and latest timestamps among displayed groups
if displayed_groups>0:
    earliest_ts = combined_display['min_timestamp'].min().strftime('%Y-%m-%d %H:%M:%S%z')
    latest_ts = combined_display['max_timestamp'].max().strftime('%Y-%m-%d %H:%M:%S%z')
else:
    earliest_ts = latest_ts = 'N/A'

# Error logs summary
if error_display.empty:
    error_summary = "No error log rows found for the allowed components."
else:
    # show top error rows (already limited to top 20 earlier)
    err_examples = []
    for _, r in error_display.iterrows():
        err_examples.append(f"{r['cmdb_id']} count={int(r['count'])} range={r['earliest_timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')} to {r['latest_timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')}")
    error_summary = "Top error log counts: " + "; ".join(err_examples[:5])

# Compose final plain-English summary string
summary = (
    f"Aggregated metric/trace/log groups (top shown): {displayed_groups} groups.\n"
    f"Most groups have count={max_count} points. Global timestamp range among displayed groups: {earliest_ts} to {latest_ts} (UTC+8).\n"
    f"Example top groups:\n - " + "\n - ".join(examples) + "\n\n"
    f"Error logs: {error_summary}"
)

# Display the summary string (compact)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_type   cmdb_id                                           kpi_name  count             min_timestamp             max_timestamp     p95_value
0     metric  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  0.000000e+00
1     metric  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  0.000000e+00
2     metric  apache01                         OSLinux-CPU_CPU_CPUSysTime     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  1.819050e+01
3     metric  apache01                            OSLinux-CPU_CPU_CPULoad     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  1.325000e+00
4     metric  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  0.000000e+00
..       ...       ...                                                ...    ...                       ...                       ...           ...
45    metric   Mysql02                         OSLinux-CPU_CPU_CPUSysTime     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  5.877260e-01
46    metric  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  0.000000e+00
47    metric      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  9.472872e+08
48    metric  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  5.000000e+02
49    metric  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31 2021-03-06 02:00:00+08:00 2021-03-06 02:30:00+08:00  0.000000e+00

[50 rows x 7 columns], Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp, latest_timestamp]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta

# Reuse allowed components and timezone if present; otherwise define
try:
    allowed_cmdb
except NameError:
    allowed_cmdb = [
        "apache01", "apache02", "Tomcat01", "Tomcat02", "Tomcat04", "Tomcat03",
        "MG01", "MG02", "IG01", "IG02", "Mysql01", "Mysql02", "Redis01", "Redis02"
    ]
try:
    tz_sh
except NameError:
    tz_sh = pytz.timezone('Asia/Shanghai')

# Ensure dataframes from kernel exist; they were loaded earlier.
# If not present, this will raise and is expected per RULE 4 to revise.
# We'll proceed assuming metric_df, trace_df, log_df, error_logs_df exist.
# Refilter to allowed components to be safe
metric_df = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()
trace_df = trace_df[trace_df['cmdb_id'].isin(allowed_cmdb)].copy()
log_df = log_df[log_df['cmdb_id'].isin(allowed_cmdb)].copy()
error_logs_df = error_logs_df[error_logs_df['cmdb_id'].isin(allowed_cmdb)].copy()

# Normalize columns: ensure timestamp numeric and value numeric
for df in (metric_df, trace_df, log_df):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Incident window in Asia/Shanghai tz
start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
end_local = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)
# Convert to UTC epoch seconds for comparing to raw timestamp (which we treat as UTC seconds)
start_utc_ts = int(start_local.tz_convert(pytz.UTC).timestamp())
end_utc_ts = int(end_local.tz_convert(pytz.UTC).timestamp())

# Helper to compute global p95 and p05 per group for a dataframe
def compute_global_thresholds(df, kpi_col, file_type_label):
    df_local = df.copy()
    df_local = df_local.rename(columns={kpi_col: 'kpi_name'})[['cmdb_id', 'kpi_name', 'value']]
    # drop NaNs in value for percentile calculations
    df_local = df_local.dropna(subset=['value'])
    if df_local.empty:
        return pd.DataFrame(columns=['file_type','cmdb_id','kpi_name','p95','p05'])
    grp = df_local.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    thresholds = grp.agg(p95 = lambda s: float(s.quantile(0.95)),
                         p05 = lambda s: float(s.quantile(0.05))).reset_index()
    thresholds['file_type'] = file_type_label
    # reorder
    thresholds = thresholds[['file_type','cmdb_id','kpi_name','p95','p05']]
    return thresholds

# Compute thresholds for metric, trace, log (global using full series)
thresholds_metric = compute_global_thresholds(metric_df, 'kpi_name', 'metric')
thresholds_trace = compute_global_thresholds(trace_df, 'trace_name', 'trace')
thresholds_log = compute_global_thresholds(log_df, 'log_name', 'log')

# Combine thresholds for easy lookup
thresholds_all = pd.concat([thresholds_metric, thresholds_trace, thresholds_log], ignore_index=True)

# Prepare function to compute in-window anomaly stats per group using precomputed thresholds
def compute_anomalies_in_window(df, kpi_col, file_type_label, thresholds_df):
    df_loc = df.copy()
    df_loc = df_loc.rename(columns={kpi_col: 'kpi_name'})[['cmdb_id','kpi_name','timestamp','value']]
    # keep only rows with numeric timestamp
    df_loc = df_loc.dropna(subset=['timestamp'])
    # Filter to incident window (inclusive)
    df_window = df_loc[(df_loc['timestamp'] >= start_utc_ts) & (df_loc['timestamp'] <= end_utc_ts)].copy()
    # Merge thresholds to windowed points to know p95/p05 per point's group
    merged = pd.merge(df_window, thresholds_df, on=['cmdb_id','kpi_name'], how='left')
    # Prepare grouping keys (use thresholds_df groups even if window empty)
    all_groups = thresholds_df[thresholds_df['file_type']==file_type_label][['cmdb_id','kpi_name','p95','p05']].copy()
    # For each group compute stats
    records = []
    for _, row in all_groups.iterrows():
        cmdb = row['cmdb_id']; kpi = row['kpi_name']; p95 = row['p95']; p05 = row['p05']
        group_points = merged[(merged['cmdb_id']==cmdb) & (merged['kpi_name']==kpi)].copy()
        count_in_window = int(group_points['value'].count())
        # Identify anomalies
        high_pts = group_points[group_points['value'] > p95] if not np.isnan(p95) else group_points.iloc[0:0]
        low_pts = group_points[group_points['value'] < p05] if not np.isnan(p05) else group_points.iloc[0:0]
        high_count = int(high_pts.shape[0])
        low_count = int(low_pts.shape[0])
        # earliest timestamps (convert to tz-aware)
        if high_count>0:
            eh_ts = int(high_pts['timestamp'].min())
            earliest_high_timestamp = pd.to_datetime(eh_ts, unit='s', utc=True).tz_convert(tz_sh)
        else:
            earliest_high_timestamp = pd.NaT
        if low_count>0:
            el_ts = int(low_pts['timestamp'].min())
            earliest_low_timestamp = pd.to_datetime(el_ts, unit='s', utc=True).tz_convert(tz_sh)
        else:
            earliest_low_timestamp = pd.NaT
        # max ratios
        if high_count>0 and p95!=0 and not np.isnan(p95):
            max_high_ratio = float((high_pts['value'] / p95).max())
        elif high_count>0 and (p95==0):
            max_high_ratio = np.inf
        else:
            max_high_ratio = np.nan
        if low_count>0 and (low_pts['value'].min()!=0) and not np.isnan(p05):
            min_low_val = float(low_pts['value'].min())
            max_low_ratio = float(p05 / min_low_val)
        elif low_count>0 and (low_pts['value'].min()==0):
            max_low_ratio = np.inf
        else:
            max_low_ratio = np.nan
        # consecutive sequences (by timestamp gaps of 60 seconds)
        def has_consec(points_df):
            if points_df.shape[0] < 2:
                return False
            ts_sorted = np.sort(points_df['timestamp'].astype(int).values)
            # compute diffs
            diffs = np.diff(ts_sorted)
            # treat consecutive if diff == 60 (one-minute sampling)
            # find longest run where consecutive diffs == 60
            if diffs.size == 0:
                return False
            runs = []
            run = 1
            for d in diffs:
                if d == 60:
                    run += 1
                else:
                    runs.append(run)
                    run = 1
            runs.append(run)
            longest = max(runs) if runs else 1
            return longest >= 2
        consecutive_high_bool = bool(has_consec(high_pts))
        consecutive_low_bool = bool(has_consec(low_pts))
        records.append({
            'file_type': file_type_label,
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'p95': p95,
            'p05': p05,
            'count_in_window': count_in_window,
            'high_count': high_count,
            'low_count': low_count,
            'earliest_high_timestamp': earliest_high_timestamp,
            'earliest_low_timestamp': earliest_low_timestamp,
            'max_high_ratio': max_high_ratio,
            'max_low_ratio': max_low_ratio,
            'consecutive_high_bool': consecutive_high_bool,
            'consecutive_low_bool': consecutive_low_bool
        })
    res = pd.DataFrame.from_records(records)
    return res

# Compute anomalies summaries per file type
anoms_metric = compute_anomalies_in_window(metric_df, 'kpi_name', 'metric', thresholds_metric)
anoms_trace = compute_anomalies_in_window(trace_df, 'trace_name', 'trace', thresholds_trace)
anoms_log   = compute_anomalies_in_window(log_df,   'log_name',   'log',   thresholds_log)

# Combine anomaly summaries
anoms_all = pd.concat([anoms_metric, anoms_trace, anoms_log], ignore_index=True)

# Filter only groups with any anomalies in-window
anoms_positive = anoms_all[(anoms_all['high_count'] > 0) | (anoms_all['low_count'] > 0)].copy()

# For compactness round ratios and limit displayed rows to top 20 by total anomalies
anoms_positive['total_anoms'] = anoms_positive['high_count'] + anoms_positive['low_count']
anoms_positive = anoms_positive.sort_values(by=['total_anoms'], ascending=False).reset_index(drop=True)
anoms_display = anoms_positive.head(20).copy()

# Round numeric ratio columns for readability
anoms_display['p95'] = anoms_display['p95'].astype(float).round(6)
anoms_display['p05'] = anoms_display['p05'].astype(float).round(6)
anoms_display['max_high_ratio'] = anoms_display['max_high_ratio'].replace(np.inf, np.nan).round(6)
anoms_display['max_low_ratio'] = anoms_display['max_low_ratio'].replace(np.inf, np.nan).round(6)

# Select and order columns as requested
anoms_display = anoms_display[[
    'file_type','cmdb_id','kpi_name','p95','p05','count_in_window',
    'high_count','low_count','earliest_high_timestamp','earliest_low_timestamp',
    'max_high_ratio','max_low_ratio','consecutive_high_bool','consecutive_low_bool'
]]

# --- Error logs: filter to incident window and allowed components (already filtered) ---
error_logs_df['timestamp'] = pd.to_numeric(error_logs_df['timestamp'], errors='coerce').astype('Int64')
err_window = error_logs_df[
    (error_logs_df['timestamp'] >= start_utc_ts) & (error_logs_df['timestamp'] <= end_utc_ts)
].copy()

# If message column exists, ensure string
if 'message' in err_window.columns:
    err_window['message'] = err_window['message'].astype(str).fillna('')
else:
    err_window['message'] = ''

# Build examples (up to 5) per cmdb_id
def build_error_examples(group):
    # sort by timestamp asc
    g = group.sort_values('timestamp')
    examples = []
    for _, r in g.head(5).iterrows():
        ts = int(r['timestamp'])
        ts_local = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz_sh)
        msg_snip = (r['message'][:120]).replace('\n',' ').replace('\r',' ')
        examples.append(f"{ts_local.strftime('%Y-%m-%d %H:%M:%S%z')}|{msg_snip}")
    return "; ".join(examples)

if err_window.empty:
    error_display = pd.DataFrame(columns=['cmdb_id','count_in_window','examples'])
else:
    err_grp = err_window.groupby('cmdb_id', dropna=False).agg(
        count_in_window = ('timestamp','count')
    ).reset_index()
    # attach examples
    examples_series = err_window.groupby('cmdb_id').apply(build_error_examples).reset_index()
    examples_series.columns = ['cmdb_id','examples']
    error_display = pd.merge(err_grp, examples_series, on='cmdb_id', how='left')
    error_display = error_display.sort_values(by='count_in_window', ascending=False).reset_index(drop=True)
    error_display = error_display.head(20)

# Final compact outputs: anomaly groups (top 20) and error log summary (top 20)
anoms_display, error_display

```
Out[2]:
```
```python
# Compose a plain-English summary using existing kernel variables (anoms_positive, anoms_display, error_display)
try:
    total_groups = int(anoms_positive.shape[0])
except NameError:
    total_groups = 0

window_str = "2021-03-06 02:00:00+08:00 to 2021-03-06 02:30:00+08:00"
lines = [f"Incident window: {window_str}", f"Total groups with any anomalies in-window: {total_groups}"]

# Top examples (up to 5) from anoms_display
try:
    top_rows = anoms_display.head(5)
    example_lines = []
    for _, r in top_rows.iterrows():
        eh = r['earliest_high_timestamp']
        el = r['earliest_low_timestamp']
        eh_s = eh.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(eh) else 'N/A'
        el_s = el.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(el) else 'N/A'
        example_lines.append(
            f"{r['file_type']} | {r['cmdb_id']} | {r['kpi_name']} | p95={r['p95']} p05={r['p05']} | "
            f"in_window={int(r['count_in_window'])} high={int(r['high_count'])} low={int(r['low_count'])} | "
            f"earliest_high={eh_s} earliest_low={el_s} | "
            f"max_high_ratio={r['max_high_ratio']} max_low_ratio={r['max_low_ratio']} | "
            f"consec_high={bool(r['consecutive_high_bool'])} consec_low={bool(r['consecutive_low_bool'])}"
        )
    if example_lines:
        lines.append("Top anomaly groups (examples):")
        lines.extend([f" - {l}" for l in example_lines])
except NameError:
    lines.append("No anomaly detail available.")

# Error logs summary in-window
try:
    if error_display.empty:
        lines.append("Error logs in-window: none found for allowed components.")
    else:
        err_lines = []
        for _, r in error_display.iterrows():
            err_lines.append(f"{r['cmdb_id']} count_in_window={int(r['count_in_window'])} examples=[{r['examples']}]")
        lines.append("Error logs in-window (top by count):")
        lines.extend([f" - {l}" for l in err_lines[:20]])
except NameError:
    lines.append("Error logs summary not available.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_type   cmdb_id                                   kpi_name           p95           p05  count_in_window  high_count  low_count   earliest_high_timestamp    earliest_low_timestamp  max_high_ratio  max_low_ratio  consecutive_high_bool  consecutive_low_bool
0        log  Tomcat03                              log.row_count  2.520000e+01  1.400000e+00               29           2          2 2021-03-06 02:03:00+08:00 2021-03-06 02:23:00+08:00        1.071429       1.400000                  False                 False
1        log  Tomcat01                              log.row_count  2.560000e+01  1.400000e+00               29           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:22:00+08:00        1.250000       1.400000                  False                  True
2     metric      IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.135891e+08  4.804165e+08               31           2          2 2021-03-06 02:29:00+08:00 2021-03-06 02:00:00+08:00        1.018304       1.034746                   True                  True
3     metric      IG01  JVM-Operating System_7778_JVM_JVM_CPULoad  7.837500e-02  5.401000e-02               22           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:18:00+08:00        1.006699       1.000185                  False                 False
4      trace  Tomcat04                 trace.to_MG02.duration_p95  4.861000e-01  3.080000e-02               25           2          2 2021-03-06 02:02:00+08:00 2021-03-06 02:16:00+08:00        1.517898       1.429234                  False                 False
5      trace  Tomcat04                trace.to_MG02.duration_mean  1.115920e-01  1.836000e-02               25           2          2 2021-03-06 02:02:00+08:00 2021-03-06 02:25:00+08:00        1.462470       1.049143                  False                 False
6      trace  Tomcat04                 trace.to_MG01.duration_p95  2.866500e-01  1.266000e-02               29           2          2 2021-03-06 02:02:00+08:00 2021-03-06 02:18:00+08:00        1.078319       1.150909                  False                  True
7      trace  Tomcat04                trace.to_MG01.duration_mean  1.115000e-01  1.140000e-02               29           2          2 2021-03-06 02:03:00+08:00 2021-03-06 02:18:00+08:00        1.488789       1.103261                  False                  True
8      trace  Tomcat04                       trace.self.row_count  1.308000e+02  8.400000e+00               29           2          2 2021-03-06 02:02:00+08:00 2021-03-06 02:18:00+08:00        1.284404       1.400000                  False                 False
9      trace  Tomcat04                    trace.self.duration_p95  5.648000e-02  1.430000e-02               29           2          2 2021-03-06 02:03:00+08:00 2021-03-06 02:18:00+08:00        2.002479       1.466667                   True                  True
10     trace  Tomcat01                 trace.to_MG02.duration_p95  5.027100e-01  1.379500e-02               27           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:17:00+08:00        1.961469       1.069380                  False                 False
11     trace  Tomcat01                    trace.to_MG02.row_count  1.380000e+01  1.300000e+00               27           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:17:00+08:00        1.159420       1.300000                  False                 False
12     trace  Tomcat02              trace.from_IG01.duration_mean  9.072300e-02  1.625000e-02               26           2          2 2021-03-06 02:15:00+08:00 2021-03-06 02:18:00+08:00        1.128311       1.083333                   True                 False
13     trace  Tomcat02               trace.from_IG01.duration_p95  2.371250e-01  1.625000e-02               26           2          2 2021-03-06 02:01:00+08:00 2021-03-06 02:18:00+08:00        1.417396       1.083333                  False                 False
14     trace  Tomcat02              trace.from_IG02.duration_mean  1.047500e-01  2.425000e-02               27           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:11:00+08:00        1.738944       1.122685                  False                 False
15     trace  Tomcat02               trace.from_IG02.duration_p95  3.428250e-01  2.861500e-02               27           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:21:00+08:00        2.135200       1.100577                  False                 False
16     trace  Tomcat01                  trace.from_IG02.row_count  1.470000e+01  1.300000e+00               27           2          2 2021-03-06 02:05:00+08:00 2021-03-06 02:23:00+08:00        1.156463       1.300000                  False                 False
17     trace  Tomcat01                   trace.self.duration_mean  2.086000e-02  3.040000e-03               29           2          2 2021-03-06 02:23:00+08:00 2021-03-06 02:18:00+08:00        1.094593       1.351111                  False                  True
18     trace  Tomcat01                    trace.self.duration_p95  5.550000e-02  1.440000e-02               29           2          2 2021-03-06 02:15:00+08:00 2021-03-06 02:19:00+08:00        1.851351       1.039711                  False                  True
19     trace  Tomcat01                       trace.self.row_count  1.640000e+02  9.200000e+00               29           2          2 2021-03-06 02:04:00+08:00 2021-03-06 02:22:00+08:00        1.371951       1.533333                  False                  True, Empty DataFrame
Columns: [cmdb_id, count_in_window, examples]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Ensure required kernel variables exist
try:
    anoms_all
except NameError:
    raise NameError("anoms_all not found in kernel. Please run previous anomaly computations first.")

# Ensure timezone and window variables; recreate if missing
try:
    tz_sh, start_local, end_local
except NameError:
    tz_sh = pytz.timezone('Asia/Shanghai')
    start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
    end_local = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)

# Work on groups that have any anomalies
anoms_candidates = anoms_all[(anoms_all['high_count'] > 0) | (anoms_all['low_count'] > 0)].copy()

# Helper to check if timestamp lies inside incident window
def in_window(ts):
    if pd.isna(ts):
        return False
    return (ts >= start_local) and (ts <= end_local)

# Mark whether earliest anomaly timestamps lie inside window
anoms_candidates['earliest_high_in_window'] = anoms_candidates['earliest_high_timestamp'].apply(in_window)
anoms_candidates['earliest_low_in_window']  = anoms_candidates['earliest_low_timestamp'].apply(in_window)

# Filter to groups with earliest anomaly inside window (either high or low)
anoms_filtered = anoms_candidates[(anoms_candidates['earliest_high_in_window']) | (anoms_candidates['earliest_low_in_window'])].copy()

# Compute severity = max(max_high_ratio (or 0), max_low_ratio (or 0))
anoms_filtered['max_high_ratio_f'] = anoms_filtered['max_high_ratio'].fillna(0)
anoms_filtered['max_low_ratio_f']  = anoms_filtered['max_low_ratio'].fillna(0)
anoms_filtered['severity'] = anoms_filtered[['max_high_ratio_f', 'max_low_ratio_f']].max(axis=1)

# Sort by severity desc and take top 20
severity_top20 = anoms_filtered.sort_values(by='severity', ascending=False).head(20).copy()

# Select and order requested columns
severity_top20 = severity_top20[[
    'file_type','cmdb_id','kpi_name','p95','p05','count_in_window',
    'high_count','low_count','earliest_high_timestamp','earliest_low_timestamp',
    'max_high_ratio','max_low_ratio','severity'
]].reset_index(drop=True)

# Compact count summary: number_of_severe_groups per cmdb_id among the top-20 severity list
severe_counts = severity_top20.groupby('cmdb_id').size().reset_index(name='number_of_severe_groups')
severe_counts = severe_counts[severe_counts['number_of_severe_groups'] > 0].sort_values(by='number_of_severe_groups', ascending=False).reset_index(drop=True)

# Store full results in kernel variables for next steps
severity_all_full = anoms_filtered.copy()
severity_top20_full = severity_top20.copy()
severe_counts_full = severe_counts.copy()

# Final compact outputs (top 20 severity rows and per-component counts)
severity_top20_full, severe_counts_full

```
Out[3]:
```
```python
# Build a plain-English summary string from existing kernel variables (severity_top20_full, severe_counts_full)
try:
    top = severity_top20_full.head(20)
    counts = severe_counts_full
except NameError:
    summary = "Required intermediate results not found in kernel."
else:
    lines = []
    lines.append("Severity-ranked candidates (incident window 2021-03-06 02:00 to 02:30 UTC+8):")
    lines.append(f"- Returned top {min(20, len(top))} groups sorted by severity (severity = max(max_high_ratio, max_low_ratio)).")
    # Note about infinite severity
    if (top['severity']==np.inf).any():
        lines.append("- Note: several groups have infinite severity because a group's in-window low anomaly had a minimum value of 0 (division by zero => infinite low ratio).")
    # top 5 examples
    lines.append("- Top example groups:")
    for _, r in top.head(5).iterrows():
        sev = r['severity']
        sev_s = "inf" if np.isinf(sev) else f"{float(sev):.6f}"
        eh = r['earliest_high_timestamp']
        el = r['earliest_low_timestamp']
        eh_s = eh.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(eh) else 'N/A'
        el_s = el.strftime('%Y-%m-%d %H:%M:%S%z') if pd.notna(el) else 'N/A'
        lines.append(f"  • {r['file_type']} | {r['cmdb_id']} | {r['kpi_name']} | severity={sev_s} | high={int(r['high_count'])} low={int(r['low_count'])} | earliest_high={eh_s} earliest_low={el_s}")
    # component-level counts
    lines.append("- Number of severe groups per component (top entries):")
    for _, r in counts.iterrows():
        lines.append(f"  • {r['cmdb_id']}: {int(r['number_of_severe_groups'])}")
    # concise conclusion
    lines.append("- Conclusion: Mysql01 is the dominant hotspot (12 severe groups), followed by apache02 and Mysql02. Many severe entries show infinite low-ratio due to zero-valued minima in-window; these should be inspected first.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_type   cmdb_id                                           kpi_name           p95          p05  count_in_window  high_count  low_count   earliest_high_timestamp    earliest_low_timestamp  max_high_ratio  max_low_ratio  severity
0     metric  apache02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps      0.443310     0.081690               15           1          1 2021-03-06 02:06:00+08:00 2021-03-06 02:27:00+08:00        1.052687            inf       inf
1     metric  apache02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite      5.640000     0.933310               15           1          1 2021-03-06 02:06:00+08:00 2021-03-06 02:27:00+08:00        1.016548            inf       inf
2     metric  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...      4.550000     0.450000               10           1          1 2021-03-06 02:17:00+08:00 2021-03-06 02:27:00+08:00        1.098901            inf       inf
3     metric      MG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  65057.800000   215.100000               10           1          1 2021-03-06 02:04:00+08:00 2021-03-06 02:26:00+08:00        1.446114            inf       inf
4     metric  Tomcat03                    OSLinux-CPU_CPU-1_SingleCpuidle     99.280845     0.643275               10           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:24:00+08:00        1.001885            inf       inf
5     metric  Tomcat03                    OSLinux-CPU_CPU-0_SingleCpuidle     99.501110     7.219440               10           1          1 2021-03-06 02:24:00+08:00 2021-03-06 02:18:00+08:00        1.000001            inf       inf
6     metric   Mysql01                  Mysql-MySQL_3306_Binlog Cache Use      0.149970     0.015865               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        3.111956            inf       inf
7     metric   Mysql01               Mysql-MySQL_3306_Innodb data written  15875.840000  2132.053365               20           1          1 2021-03-06 02:29:00+08:00 2021-03-06 02:19:00+08:00        1.065871            inf       inf
8     metric   Mysql01        Mysql-MySQL_3306_Innodb dblwr pages written      0.468365     0.063365               20           1          1 2021-03-06 02:29:00+08:00 2021-03-06 02:19:00+08:00        1.067543            inf       inf
9     metric   Mysql01               Mysql-MySQL_3306_Innodb dblwr writes      0.300835     0.047500               20           1          1 2021-03-06 02:03:00+08:00 2021-03-06 02:19:00+08:00        1.052737            inf       inf
10    metric   Mysql01         Mysql-MySQL_3306_Innodb log write requests      0.185030     0.015865               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        2.882235            inf       inf
11    metric   Mysql01                 Mysql-MySQL_3306_Innodb log writes      0.453335     0.063365               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        1.139775            inf       inf
12    metric   Mysql01              Mysql-MySQL_3306_Innodb os log fsyncs      0.650000     0.095000               20           0          1                       NaT 2021-03-06 02:19:00+08:00             NaN            inf       inf
13    metric   Mysql01             Mysql-MySQL_3306_Innodb os log written    450.986665    40.533365               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        1.665090            inf       inf
14    metric   Mysql01                    Mysql-MySQL_3306_Handler Update      0.136635     0.015865               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        1.463754            inf       inf
15    metric   Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages flushed      0.468365     0.063365               20           1          1 2021-03-06 02:29:00+08:00 2021-03-06 02:19:00+08:00        1.067543            inf       inf
16    metric   Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...      1.220800     0.110865               20           1          1 2021-03-06 02:00:00+08:00 2021-03-06 02:19:00+08:00        3.918168            inf       inf
17    metric   Mysql02              Mysql-MySQL_3306_Innodb pages created      0.599970     0.063365               20           1          1 2021-03-06 02:05:00+08:00 2021-03-06 02:19:00+08:00        1.527910            inf       inf
18    metric   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKRea...     92.500000     5.000000               11           1          1 2021-03-06 02:04:00+08:00 2021-03-06 02:26:00+08:00        1.524324            inf       inf
19    metric   Mysql02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite   2853.666667     0.560000               15           1          1 2021-03-06 02:03:00+08:00 2021-03-06 02:27:00+08:00        1.043009            inf       inf,     cmdb_id  number_of_severe_groups
0   Mysql01                       12
1  apache02                        3
2   Mysql02                        2
3  Tomcat03                        2
4      MG01                        1)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel where available
try:
    anoms_all
except NameError:
    raise NameError("anoms_all not found in kernel. Please run prior anomaly computations.")

# Ensure timezone and incident window variables
try:
    tz_sh, start_local, end_local
except NameError:
    tz_sh = pytz.timezone('Asia/Shanghai')
    start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
    end_local = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)

# Work on groups that have any anomalies (high or low)
df = anoms_all[(anoms_all['high_count'] > 0) | (anoms_all['low_count'] > 0)].copy()

# Ensure earliest_high_timestamp and earliest_low_timestamp are timezone-aware Timestamps (they should be)
# Compute anomaly_timestamp and anomaly_type per rules:
def pick_earliest_anom(row):
    eh = row.get('earliest_high_timestamp')
    el = row.get('earliest_low_timestamp')
    # if both NaT -> return (NaT, None)
    if pd.isna(eh) and pd.isna(el):
        return pd.NaT, None
    # if only eh exists
    if pd.notna(eh) and pd.isna(el):
        return eh, 'high'
    # if only el exists
    if pd.isna(eh) and pd.notna(el):
        return el, 'low'
    # both exist: if equal prefer high
    if eh == el:
        return eh, 'high'
    # otherwise pick the earlier timestamp
    if eh < el:
        return eh, 'high'
    else:
        return el, 'low'

picked = df.apply(lambda r: pick_earliest_anom(r), axis=1)
df['anomaly_timestamp'] = [t for t,_ in picked]
df['anomaly_type'] = [typ for _,typ in picked]

# Filter to those with anomaly_timestamp inside the incident window (just in case)
df = df[df['anomaly_timestamp'].notna()].copy()
df = df[(df['anomaly_timestamp'] >= start_local) & (df['anomaly_timestamp'] <= end_local)].copy()

# Compute severity if not present: severity = max(max_high_ratio (or 0), max_low_ratio (or 0))
df['max_high_ratio_f'] = df['max_high_ratio'].fillna(0)
df['max_low_ratio_f']  = df['max_low_ratio'].fillna(0)
df['severity'] = df[['max_high_ratio_f','max_low_ratio_f']].max(axis=1)

# Select and order columns requested
cols = [
    'file_type','cmdb_id','kpi_name','anomaly_type','anomaly_timestamp','p95','p05',
    'count_in_window','high_count','low_count','max_high_ratio','max_low_ratio',
    'consecutive_high_bool','consecutive_low_bool','severity'
]
# Ensure all cols exist
for c in cols:
    if c not in df.columns:
        df[c] = np.nan

# Sort by anomaly_timestamp ascending and take top 20
earliest_anoms_sorted = df.sort_values(by='anomaly_timestamp', ascending=True).reset_index(drop=True)
earliest_top20 = earliest_anoms_sorted.head(20).copy()

# Component-level earliest anomaly among the top-20 groups
components = earliest_top20.groupby('cmdb_id', as_index=False).agg(
    earliest_anomaly_timestamp_for_component = ('anomaly_timestamp', 'min')
).sort_values(by='earliest_anomaly_timestamp_for_component', ascending=True).reset_index(drop=True)

# Store full results in kernel variables for next steps
earliest_anoms_full = earliest_anoms_sorted.copy()
earliest_anoms_top20 = earliest_top20.copy()
component_earliest_summary_full = components.copy()

# Keep outputs compact: display selected columns only (top 20) and component summary
# Round numeric columns for compactness
for col in ['p95','p05','max_high_ratio','max_low_ratio','severity']:
    if col in earliest_anoms_top20.columns:
        earliest_anoms_top20[col] = earliest_anoms_top20[col].apply(lambda x: (float('inf') if np.isinf(x) else (np.nan if pd.isna(x) else round(float(x),6))))

earliest_anoms_top20[cols], components

```
Out[4]:
```
```python
# Build a concise plain-English summary from the existing kernel results (earliest_anoms_top20, component_earliest_summary_full)
try:
    top = earliest_anoms_top20
    comp_earliest = component_earliest_summary_full
except NameError:
    summary = "Required results not found in kernel. Please run the anomaly extraction steps first."
else:
    n_groups = len(top)
    lines = []
    lines.append("Earliest in-window anomalies (incident window 2021-03-06 02:00:00 to 02:30:00 UTC+8)")
    lines.append(f"- Returned groups: {n_groups} (showing top {min(20,n_groups)} by earliest anomaly).")
    # Top 3 earliest entries as examples
    lines.append("- Top earliest anomaly examples:")
    for _, r in top.head(3).iterrows():
        ts = r['anomaly_timestamp'].strftime('%Y-03-%d %H:%M:%S%z') if hasattr(r['anomaly_timestamp'], 'strftime') else str(r['anomaly_timestamp'])
        lines.append(f"  • {r['file_type']} | {r['cmdb_id']} | {r['kpi_name']} | {r['anomaly_type']} at {r['anomaly_timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')} | high_count={int(r['high_count'])} low_count={int(r['low_count'])} | severity={r['severity']}")
    # Key observations
    lines.append("- Key observations:")
    lines.append("  1) The earliest anomaly (02:00+08) is on IG01: JVM heap memory (low anomaly).")
    lines.append("  2) Multiple Redis metrics (Redis01 and Redis02) show anomalies starting at 02:00+08 across many Redis-related KPIs (several low anomalies, many with consecutive flags true).")
    lines.append("  3) Mysql02 also has early low anomalies at 02:00+08 for several metrics.")
    # Component-level earliest timestamps
    lines.append("- Earliest anomaly timestamp per affected component (from the top-20 list):")
    for _, r in comp_earliest.iterrows():
        lines.append(f"  • {r['cmdb_id']}: {r['earliest_anomaly_timestamp_for_component'].strftime('%Y-%m-%d %H:%M:%S%z')}")
    # Actionable prioritization
    lines.append("- Recommended prioritization:")
    lines.append("  1) Investigate IG01 (heap memory anomaly at 02:00) first.")
    lines.append("  2) Next check Redis01 and Redis02 (multiple Redis metrics anomalous at 02:00, some consecutive).")
    lines.append("  3) Then inspect Mysql02-related anomalies.")
    lines.append("  Note: inspect any zero-valued minima that produced infinite low-ratio severities when assessing root cause.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   file_type  cmdb_id                                           kpi_name anomaly_type         anomaly_timestamp           p95           p05  count_in_window  high_count  low_count  max_high_ratio  max_low_ratio  consecutive_high_bool  consecutive_low_bool  severity
0     metric     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage          low 2021-03-06 02:00:00+08:00  3.470060e+01  2.275750e+01               21           1          1        1.015962       1.017113                  False                 False  1.017113
1     metric  Redis01                             OSLinux-CPU_CPU_CPUWio         high 2021-03-06 02:00:00+08:00  3.681040e-01  4.200000e-03               29           2          1        1.219078       1.010830                  False                 False  1.219078
2     metric  Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime          low 2021-03-06 02:00:00+08:00  6.168532e+06  6.167013e+06               29           2          2        1.000014       1.000014                   True                  True  1.000014
3     metric  Redis01            redis-Redis_6379_Redis  (keyspace_hits)          low 2021-03-06 02:00:00+08:00  3.309159e+09  3.309070e+09               30           2          2        1.000002       1.000002                   True                  True  1.000002
4     metric  Redis01                redis-Redis_6379_Redis  (lru_clock)          low 2021-03-06 02:00:00+08:00  4.356118e+06  4.354552e+06               30           2          2        1.000020       1.000020                   True                  True  1.000020
5     metric  Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...          low 2021-03-06 02:00:00+08:00  9.830137e+08  9.829794e+08               30           2          2        1.000002       1.000002                   True                  True  1.000002
6     metric  Redis01  redis-Redis_6379_Redis  (total_commands_proces...          low 2021-03-06 02:00:00+08:00  4.381791e+09  4.381635e+09               30           2          2        1.000002       1.000002                   True                  True  1.000002
7     metric  Redis01  redis-Redis_6379_Redis  (total_connections_rec...          low 2021-03-06 02:00:00+08:00  2.996735e+07  2.995683e+07               30           2          2        1.000020       1.000018                   True                  True  1.000020
8     metric  Redis01        redis-Redis_6379_Redis  (uptime_in_seconds)          low 2021-03-06 02:00:00+08:00  4.603776e+06  4.602210e+06               30           2          2        1.000019       1.000019                   True                  True  1.000019
9     metric  Redis01             redis-Redis_6379_Redis  (used_cpu_sys)          low 2021-03-06 02:00:00+08:00  7.026190e+04  7.025860e+04               30           2          2        1.000004       1.000003                   True                  True  1.000004
10    metric  Redis01            redis-Redis_6379_Redis  (used_cpu_user)          low 2021-03-06 02:00:00+08:00  3.051121e+04  3.050874e+04               30           2          2        1.000002       1.000004                   True                  True  1.000004
11    metric  Redis01          redis-Redis_6379_Redis  (used_memory_rss)          low 2021-03-06 02:00:00+08:00  1.215754e+07  1.155850e+07               30           2          2        1.003655       1.005308                   True                  True  1.005308
12    metric  Redis02                             OSLinux-CPU_CPU_CPUWio         high 2021-03-06 02:00:00+08:00  3.512000e-01  4.156000e-03               31           2          0        1.609895            NaN                  False                 False  1.609895
13    metric  Redis02        OSLinux-NTP_197.30.1.67_NtpServerTimeOffset          low 2021-03-06 02:00:00+08:00  7.000000e-03 -1.000000e-02                5           1          1        1.142857       0.909091                  False                 False  1.142857
14    metric  Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...          low 2021-03-06 02:00:00+08:00  1.630000e+00  1.623000e+00                7           0          1             NaN       1.001852                  False                 False  1.001852
15    metric  Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime          low 2021-03-06 02:00:00+08:00  4.618911e+06  4.617286e+06               31           2          2        1.000020       1.000019                   True                  True  1.000020
16    metric  Redis02            redis-Redis_6379_Redis  (keyspace_hits)          low 2021-03-06 02:00:00+08:00  2.577051e+09  2.576992e+09               30           2          2        1.000002       1.000001                   True                  True  1.000002
17    metric  Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las...          low 2021-03-06 02:00:00+08:00  8.013065e+08  8.012837e+08               30           2          2        1.000002       1.000002                   True                  True  1.000002
18    metric  Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime          low 2021-03-06 02:00:00+08:00  6.167521e+06  6.165896e+06               31           2          2        1.000015       1.000015                   True                  True  1.000015
19    metric  Mysql02             Mysql-MySQL_3306_Table Locks Immediate          low 2021-03-06 02:00:00+08:00  5.080000e-02  4.840500e-02               20           0          1             NaN       1.453604                  False                 False  1.453604,    cmdb_id earliest_anomaly_timestamp_for_component
0     IG01                2021-03-06 02:00:00+08:00
1  Mysql02                2021-03-06 02:00:00+08:00
2  Redis01                2021-03-06 02:00:00+08:00
3  Redis02                2021-03-06 02:00:00+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse variables from kernel
try:
    trace_df
    thresholds_trace
    earliest_anoms_top20
except NameError as e:
    raise NameError("Required variables not found in kernel: " + str(e))

# Timezone and incident window (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
end_local   = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)
start_utc_ts = int(start_local.tz_convert(pytz.UTC).timestamp())
end_utc_ts   = int(end_local.tz_convert(pytz.UTC).timestamp())

# Identify trace groups present in earliest_anoms_top20
trace_groups = earliest_anoms_top20[earliest_anoms_top20['file_type']=='trace'][['cmdb_id','kpi_name']].drop_duplicates()
trace_groups = trace_groups.rename(columns={'kpi_name':'trace_name'}).reset_index(drop=True)

# Prepare results container
results = []

# Ensure numeric columns in trace_df
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Iterate groups and find earliest breach in-window using precomputed thresholds
for _, g in trace_groups.iterrows():
    cmdb = g['cmdb_id']
    tname = g['trace_name']
    # lookup thresholds for this trace group
    thr = thresholds_trace[(thresholds_trace['cmdb_id']==cmdb) & (thresholds_trace['kpi_name']==tname)]
    if thr.empty:
        # no global threshold available, skip
        continue
    p95 = float(thr['p95'].iloc[0])
    p05 = float(thr['p05'].iloc[0])
    # filter raw trace rows for this group in incident window
    grp_rows = trace_df[(trace_df['cmdb_id']==cmdb) & (trace_df['trace_name']==tname) &
                        (trace_df['timestamp'] >= start_utc_ts) & (trace_df['timestamp'] <= end_utc_ts)].copy()
    if grp_rows.empty:
        continue
    # sort by timestamp ascending
    grp_rows = grp_rows.sort_values('timestamp')
    # drop rows with NaN value
    grp_rows = grp_rows[grp_rows['value'].notna()].copy()
    if grp_rows.empty:
        continue
    # find earliest row where value > p95 or value < p05
    breach_idx = None
    breach_type = None
    breach_ratio = None
    breach_row = None
    for _, r in grp_rows.iterrows():
        val = float(r['value'])
        if (not np.isnan(p95)) and (val > p95):
            breach_idx = r['timestamp']
            breach_type = 'high'
            # compute ratio value/p95, handle p95==0
            if p95 == 0:
                breach_ratio = np.inf
            else:
                breach_ratio = val / p95
            breach_row = r
            break
        if (not np.isnan(p05)) and (val < p05):
            breach_idx = r['timestamp']
            breach_type = 'low'
            # compute ratio p05/value, handle value==0
            if val == 0:
                breach_ratio = np.inf
            else:
                breach_ratio = p05 / val
            breach_row = r
            break
    if breach_row is None:
        continue
    # compose result record
    ts = int(breach_row['timestamp'])
    ts_dt = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz_sh)
    results.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'timestamp': ts_dt,
        'value': float(breach_row['value']),
        'p95': p95,
        'p05': p05,
        'anomaly_type': breach_type,
        'ratio': float(breach_ratio) if not np.isinf(breach_ratio) else np.inf
    })

# Create DataFrame of earliest-anomaly trace rows
trace_earliest_anoms = pd.DataFrame(results)

# Sort by timestamp ascending and limit to top 50 for display
if not trace_earliest_anoms.empty:
    trace_earliest_anoms = trace_earliest_anoms.sort_values('timestamp').reset_index(drop=True)
    # Prepare compact display (only requested columns), limit to 50
    display_cols = ['cmdb_id','trace_name','timestamp','value','p95','p05','anomaly_type','ratio']
    trace_earliest_anoms_display = trace_earliest_anoms[display_cols].head(50).copy()
    # Round numeric columns for compactness, keep inf as-is
    for c in ['value','p95','p05','ratio']:
        trace_earliest_anoms_display[c] = trace_earliest_anoms_display[c].apply(
            lambda x: (float('inf') if np.isinf(x) else (np.nan if pd.isna(x) else round(float(x),6)))
        )
else:
    # empty results
    trace_earliest_anoms_display = pd.DataFrame(columns=['cmdb_id','trace_name','timestamp','value','p95','p05','anomaly_type','ratio'])

# Store full extracted earliest-anomaly trace rows in kernel variable trace_earliest_anoms (already set)
# Final compact output: trace_earliest_anoms_display
trace_earliest_anoms_display

```
Out[5]:
```
```python
# Plain-English summary of the trace earliest-anomaly extraction result
try:
    trace_earliest_anoms  # previously stored variable
    empty = trace_earliest_anoms.empty
except NameError:
    summary = "Trace earliest-anomaly results not found in kernel."
else:
    if empty:
        summary = (
            "No trace earliest-anomaly rows found.\n\n"
            "Explanation: For the trace groups that appeared in the earliest-anoms table, "
            "there were no raw trace points within the incident window (2021-03-06 02:00:00+08:00 to 2021-03-06 02:30:00+08:00) "
            "that breached their group's global P95 (high) or P05 (low) thresholds. "
            "Therefore there are no single earliest trace-anomaly rows to report."
        )
    else:
        # If not empty (unexpected), summarize count and earliest example
        cnt = len(trace_earliest_anoms)
        first = trace_earliest_anoms.sort_values('timestamp').iloc[0]
        ts = first['timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')
        summary = (
            f"Found {cnt} earliest trace-anomaly rows. "
            f"Earliest: {first['cmdb_id']} | {first['trace_name']} | {first['anomaly_type']} at {ts} | ratio={first['ratio']}"
        )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, timestamp, value, p95, p05, anomaly_type, ratio]
Index: []```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables: metric_df, thresholds_metric, earliest_anoms_top20
# Timezone and incident window (UTC+8)
tz_sh = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
end_local   = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)
start_utc_ts = int(start_local.tz_convert(pytz.UTC).timestamp())
end_utc_ts   = int(end_local.tz_convert(pytz.UTC).timestamp())

# Ensure numeric types in metric_df
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Identify metric groups from earliest_anoms_top20
metric_groups = earliest_anoms_top20[earliest_anoms_top20['file_type']=='metric'][['cmdb_id','kpi_name']].drop_duplicates().reset_index(drop=True)

# Helper to detect consecutive sequences (>=2) for boolean mask on timestamps (diff==60)
def has_consecutive_points(ts_array):
    if len(ts_array) < 2:
        return False
    ts_sorted = np.sort(ts_array.astype(int))
    diffs = np.diff(ts_sorted)
    # any run of diffs==60 of length >=1 corresponds to consecutive points count >=2
    run = 1
    longest = 1
    for d in diffs:
        if d == 60:
            run += 1
        else:
            if run > longest:
                longest = run
            run = 1
    if run > longest:
        longest = run
    return longest >= 2

records = []

# Iterate metric groups and find earliest breach in-window using precomputed thresholds_metric
for _, g in metric_groups.iterrows():
    cmdb = g['cmdb_id']
    kpi = g['kpi_name']
    # lookup thresholds (global) for this group
    thr = thresholds_metric[(thresholds_metric['cmdb_id']==cmdb) & (thresholds_metric['kpi_name']==kpi)]
    if thr.empty:
        continue
    p95 = float(thr['p95'].iloc[0])
    p05 = float(thr['p05'].iloc[0])
    # filter raw metric rows for this group in incident window
    grp_rows = metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==kpi) &
                         (metric_df['timestamp'] >= start_utc_ts) & (metric_df['timestamp'] <= end_utc_ts)].copy()
    if grp_rows.empty:
        continue
    grp_rows = grp_rows.dropna(subset=['value']).sort_values('timestamp')
    if grp_rows.empty:
        continue
    # counts
    high_mask = (~grp_rows['value'].isna()) & (grp_rows['value'] > p95) if not np.isnan(p95) else pd.Series(False, index=grp_rows.index)
    low_mask  = (~grp_rows['value'].isna()) & (grp_rows['value'] < p05) if not np.isnan(p05) else pd.Series(False, index=grp_rows.index)
    high_count_in_window = int(high_mask.sum())
    low_count_in_window  = int(low_mask.sum())
    # earliest timestamps for each type
    earliest_high_ts = int(grp_rows.loc[high_mask, 'timestamp'].min()) if high_count_in_window>0 else None
    earliest_low_ts  = int(grp_rows.loc[low_mask, 'timestamp'].min())  if low_count_in_window>0  else None
    # determine earliest breach according to rules (prefer high if equal)
    if earliest_high_ts is None and earliest_low_ts is None:
        continue  # no breach in-window
    if earliest_high_ts is not None and earliest_low_ts is not None:
        if earliest_high_ts <= earliest_low_ts:
            chosen_ts = earliest_high_ts
            anomaly_type = 'high'
        else:
            chosen_ts = earliest_low_ts
            anomaly_type = 'low'
    elif earliest_high_ts is not None:
        chosen_ts = earliest_high_ts
        anomaly_type = 'high'
    else:
        chosen_ts = earliest_low_ts
        anomaly_type = 'low'
    # extract the breach row(s) at chosen_ts; if both high and low at same timestamp prefer high
    rows_at_ts = grp_rows[grp_rows['timestamp'] == chosen_ts]
    # determine actual value and anomaly_type if both present (prefer high)
    val = float(rows_at_ts['value'].iloc[0])
    if anomaly_type == 'high':
        # ensure this timestamp indeed has high; if not, but low exists, adjust
        if not (val > p95):
            if val < p05:
                anomaly_type = 'low'
            else:
                # no breach at chosen ts (edge) -> find next earliest breach of chosen type
                if high_count_in_window>0:
                    chosen_ts = int(grp_rows.loc[high_mask,'timestamp'].min())
                    val = float(grp_rows.loc[grp_rows['timestamp']==chosen_ts,'value'].iloc[0])
                    anomaly_type = 'high'
                elif low_count_in_window>0:
                    chosen_ts = int(grp_rows.loc[low_mask,'timestamp'].min())
                    val = float(grp_rows.loc[grp_rows['timestamp']==chosen_ts,'value'].iloc[0])
                    anomaly_type = 'low'
                else:
                    continue
    else:
        if not (val < p05):
            if val > p95:
                anomaly_type = 'high'
            else:
                if low_count_in_window>0:
                    chosen_ts = int(grp_rows.loc[low_mask,'timestamp'].min())
                    val = float(grp_rows.loc[grp_rows['timestamp']==chosen_ts,'value'].iloc[0])
                    anomaly_type = 'low'
                elif high_count_in_window>0:
                    chosen_ts = int(grp_rows.loc[high_mask,'timestamp'].min())
                    val = float(grp_rows.loc[grp_rows['timestamp']==chosen_ts,'value'].iloc[0])
                    anomaly_type = 'high'
                else:
                    continue
    # compute ratio
    if anomaly_type == 'high':
        if p95 == 0:
            ratio = np.inf
        else:
            ratio = val / p95
    else:  # low
        if val == 0:
            ratio = np.inf
        else:
            ratio = p05 / val
    # consecutive booleans for the group within window
    high_consec = has_consecutive_points(grp_rows.loc[high_mask, 'timestamp'].values) if high_count_in_window>0 else False
    low_consec  = has_consecutive_points(grp_rows.loc[low_mask, 'timestamp'].values)  if low_count_in_window>0  else False
    # convert chosen_ts to timezone-aware datetime
    ts_dt = pd.to_datetime(int(chosen_ts), unit='s', utc=True).tz_convert(tz_sh)
    records.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'timestamp': ts_dt,
        'value': val,
        'p95': p95,
        'p05': p05,
        'anomaly_type': anomaly_type,
        'ratio': ratio,
        'high_count_in_window': high_count_in_window,
        'low_count_in_window': low_count_in_window,
        'consecutive_high_bool': bool(high_consec),
        'consecutive_low_bool': bool(low_consec)
    })

# Build DataFrame and store full result
metric_earliest_anoms = pd.DataFrame.from_records(records)

# Sort by timestamp ascending and limit display to 50 rows
if not metric_earliest_anoms.empty:
    metric_earliest_anoms = metric_earliest_anoms.sort_values('timestamp').reset_index(drop=True)
    display_cols = ['cmdb_id','kpi_name','timestamp','value','p95','p05','anomaly_type','ratio',
                    'high_count_in_window','low_count_in_window','consecutive_high_bool','consecutive_low_bool']
    metric_earliest_anoms_display = metric_earliest_anoms[display_cols].head(50).copy()
    # Round numeric columns for compactness, keep inf as-is
    for c in ['value','p95','p05','ratio']:
        metric_earliest_anoms_display[c] = metric_earliest_anoms_display[c].apply(
            lambda x: (float('inf') if np.isinf(x) else (np.nan if pd.isna(x) else round(float(x),6)))
        )
else:
    metric_earliest_anoms_display = pd.DataFrame(columns=['cmdb_id','kpi_name','timestamp','value','p95','p05','anomaly_type','ratio',
                                                          'high_count_in_window','low_count_in_window','consecutive_high_bool','consecutive_low_bool'])

# Store full DataFrame metric_earliest_anoms in kernel for later steps
metric_earliest_anoms, metric_earliest_anoms_display

```
Out[6]:
```
```python
# Summarize metric earliest anomalies in plain English using kernel-stored results
try:
    df = metric_earliest_anoms  # full results
except NameError:
    summary = "Metric earliest-anomaly results not found in kernel."
else:
    if df.empty:
        summary = "No metric earliest-anomaly rows found within the incident window."
    else:
        # basic counts and earliest timestamp
        n_groups = int(df.shape[0])
        earliest = df['timestamp'].min().strftime('%Y-%m-%d %H:%M:%S%z')
        latest = df['timestamp'].max().strftime('%Y-%m-%d %H:%M:%S%z')
        lines = []
        lines.append(f"Metric earliest anomalies: {n_groups} groups with breaches in-window (window 2021-03-06 02:00–02:30 UTC+8).")
        lines.append(f"Anomaly time range (in-window): {earliest} to {latest}.")
        # Top 6 example groups (compact)
        lines.append("Top example earliest anomalies (cmdb | KPI | type | time | ratio | high_count/low_count | consecutive_high/low):")
        ex = df.sort_values('timestamp').head(6)
        for _, r in ex.iterrows():
            ts = r['timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')
            ratio = "inf" if np.isinf(r['ratio']) else f"{float(r['ratio']):.6f}"
            lines.append(
                f" - {r['cmdb_id']} | {r['kpi_name']} | {r['anomaly_type']} | {ts} | ratio={ratio} "
                f"| {int(r['high_count_in_window'])}/{int(r['low_count_in_window'])} "
                f"| consec_high={bool(r['consecutive_high_bool'])} consec_low={bool(r['consecutive_low_bool'])}"
            )
        # Key observations and prioritization
        lines.append("Key observations / prioritization:")
        lines.append("  1) IG01 shows a JVM heap memory low anomaly at 02:00 (ratio ~1.017) — inspect heap usage and GC first.")
        lines.append("  2) Redis nodes (Redis02 and Redis01) have many Redis-related metrics breaching at 02:00; several of these show consecutive anomalies — investigate Redis health and underlying host resources next.")
        lines.append("  3) Mysql02 shows low anomalies (including 'Table Locks Immediate') at 02:00; check MySQL locks and related I/O.")
        lines.append("  4) Ratios for many metric breaches are near 1.0 (small deviations), but some entries (e.g., MySQL I/O/log related KPIs seen earlier in severity ranking) show larger ratios and should be checked after the above.")
        lines.append("  5) Use the above earliest-anomaly rows to inspect logs and traces around those timestamps for root-cause clues.")
        summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name                 timestamp         value           p95           p05 anomaly_type     ratio  high_count_in_window  low_count_in_window  consecutive_high_bool  consecutive_low_bool
0      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage 2021-03-06 02:00:00+08:00  2.237460e+01  3.470060e+01  2.275750e+01          low  1.017113                     1                    1                  False                 False
1   Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las... 2021-03-06 02:00:00+08:00  8.012823e+08  8.013065e+08  8.012837e+08          low  1.000002                     2                    2                   True                  True
2   Redis02            redis-Redis_6379_Redis  (keyspace_hits) 2021-03-06 02:00:00+08:00  2.576989e+09  2.577051e+09  2.576992e+09          low  1.000001                     2                    2                   True                  True
3   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  4.617196e+06  4.618911e+06  4.617286e+06          low  1.000019                     2                    2                   True                  True
4   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg... 2021-03-06 02:00:00+08:00  1.620000e+00  1.630000e+00  1.623000e+00          low  1.001852                     0                    1                  False                 False
5   Redis02        OSLinux-NTP_197.30.1.67_NtpServerTimeOffset 2021-03-06 02:00:00+08:00 -1.100000e-02  7.000000e-03 -1.000000e-02          low  0.909091                     1                    1                  False                 False
6   Redis02                             OSLinux-CPU_CPU_CPUWio 2021-03-06 02:00:00+08:00  4.364000e-01  3.512000e-01  4.156000e-03         high  1.242597                     2                    0                  False                 False
7   Redis01          redis-Redis_6379_Redis  (used_memory_rss) 2021-03-06 02:00:00+08:00  1.155482e+07  1.215754e+07  1.155850e+07          low  1.000319                     2                    2                   True                  True
8   Redis01            redis-Redis_6379_Redis  (used_cpu_user) 2021-03-06 02:00:00+08:00  3.050864e+04  3.051121e+04  3.050874e+04          low  1.000004                     2                    2                   True                  True
9   Redis01             redis-Redis_6379_Redis  (used_cpu_sys) 2021-03-06 02:00:00+08:00  7.025837e+04  7.026190e+04  7.025860e+04          low  1.000003                     2                    2                   True                  True
10  Redis01        redis-Redis_6379_Redis  (uptime_in_seconds) 2021-03-06 02:00:00+08:00  4.602123e+06  4.603776e+06  4.602210e+06          low  1.000019                     2                    2                   True                  True
11  Redis01  redis-Redis_6379_Redis  (total_connections_rec... 2021-03-06 02:00:00+08:00  2.995627e+07  2.996735e+07  2.995683e+07          low  1.000018                     2                    2                   True                  True
12  Redis01  redis-Redis_6379_Redis  (total_commands_proces... 2021-03-06 02:00:00+08:00  4.381626e+09  4.381791e+09  4.381635e+09          low  1.000002                     2                    2                   True                  True
13  Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las... 2021-03-06 02:00:00+08:00  9.829773e+08  9.830137e+08  9.829794e+08          low  1.000002                     2                    2                   True                  True
14  Redis01                redis-Redis_6379_Redis  (lru_clock) 2021-03-06 02:00:00+08:00  4.354465e+06  4.356118e+06  4.354552e+06          low  1.000020                     2                    2                   True                  True
15  Redis01            redis-Redis_6379_Redis  (keyspace_hits) 2021-03-06 02:00:00+08:00  3.309064e+09  3.309159e+09  3.309070e+09          low  1.000002                     2                    2                   True                  True
16  Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  6.166929e+06  6.168532e+06  6.167013e+06          low  1.000014                     2                    2                   True                  True
17  Redis01                             OSLinux-CPU_CPU_CPUWio 2021-03-06 02:00:00+08:00  4.279360e-01  3.681036e-01  4.200000e-03         high  1.162542                     2                    1                  False                 False
18  Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  6.165806e+06  6.167521e+06  6.165896e+06          low  1.000015                     2                    2                   True                  True
19  Mysql02             Mysql-MySQL_3306_Table Locks Immediate 2021-03-06 02:00:00+08:00  3.330000e-02  5.080000e-02  4.840500e-02          low  1.453604                     0                    1                  False                 False,     cmdb_id                                           kpi_name                 timestamp         value           p95           p05 anomaly_type     ratio  high_count_in_window  low_count_in_window  consecutive_high_bool  consecutive_low_bool
0      IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage 2021-03-06 02:00:00+08:00  2.237460e+01  3.470060e+01  2.275750e+01          low  1.017113                     1                    1                  False                 False
1   Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las... 2021-03-06 02:00:00+08:00  8.012823e+08  8.013065e+08  8.012837e+08          low  1.000002                     2                    2                   True                  True
2   Redis02            redis-Redis_6379_Redis  (keyspace_hits) 2021-03-06 02:00:00+08:00  2.576989e+09  2.577051e+09  2.576992e+09          low  1.000001                     2                    2                   True                  True
3   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  4.617196e+06  4.618911e+06  4.617286e+06          low  1.000019                     2                    2                   True                  True
4   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg... 2021-03-06 02:00:00+08:00  1.620000e+00  1.630000e+00  1.623000e+00          low  1.001852                     0                    1                  False                 False
5   Redis02        OSLinux-NTP_197.30.1.67_NtpServerTimeOffset 2021-03-06 02:00:00+08:00 -1.100000e-02  7.000000e-03 -1.000000e-02          low  0.909091                     1                    1                  False                 False
6   Redis02                             OSLinux-CPU_CPU_CPUWio 2021-03-06 02:00:00+08:00  4.364000e-01  3.512000e-01  4.156000e-03         high  1.242597                     2                    0                  False                 False
7   Redis01          redis-Redis_6379_Redis  (used_memory_rss) 2021-03-06 02:00:00+08:00  1.155482e+07  1.215754e+07  1.155850e+07          low  1.000319                     2                    2                   True                  True
8   Redis01            redis-Redis_6379_Redis  (used_cpu_user) 2021-03-06 02:00:00+08:00  3.050864e+04  3.051121e+04  3.050874e+04          low  1.000004                     2                    2                   True                  True
9   Redis01             redis-Redis_6379_Redis  (used_cpu_sys) 2021-03-06 02:00:00+08:00  7.025837e+04  7.026190e+04  7.025860e+04          low  1.000003                     2                    2                   True                  True
10  Redis01        redis-Redis_6379_Redis  (uptime_in_seconds) 2021-03-06 02:00:00+08:00  4.602123e+06  4.603776e+06  4.602210e+06          low  1.000019                     2                    2                   True                  True
11  Redis01  redis-Redis_6379_Redis  (total_connections_rec... 2021-03-06 02:00:00+08:00  2.995627e+07  2.996735e+07  2.995683e+07          low  1.000018                     2                    2                   True                  True
12  Redis01  redis-Redis_6379_Redis  (total_commands_proces... 2021-03-06 02:00:00+08:00  4.381626e+09  4.381791e+09  4.381635e+09          low  1.000002                     2                    2                   True                  True
13  Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las... 2021-03-06 02:00:00+08:00  9.829773e+08  9.830137e+08  9.829794e+08          low  1.000002                     2                    2                   True                  True
14  Redis01                redis-Redis_6379_Redis  (lru_clock) 2021-03-06 02:00:00+08:00  4.354465e+06  4.356118e+06  4.354552e+06          low  1.000020                     2                    2                   True                  True
15  Redis01            redis-Redis_6379_Redis  (keyspace_hits) 2021-03-06 02:00:00+08:00  3.309064e+09  3.309159e+09  3.309070e+09          low  1.000002                     2                    2                   True                  True
16  Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  6.166929e+06  6.168532e+06  6.167013e+06          low  1.000014                     2                    2                   True                  True
17  Redis01                             OSLinux-CPU_CPU_CPUWio 2021-03-06 02:00:00+08:00  4.279360e-01  3.681040e-01  4.200000e-03         high  1.162542                     2                    1                  False                 False
18  Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime 2021-03-06 02:00:00+08:00  6.165806e+06  6.167521e+06  6.165896e+06          low  1.000015                     2                    2                   True                  True
19  Mysql02             Mysql-MySQL_3306_Table Locks Immediate 2021-03-06 02:00:00+08:00  3.330000e-02  5.080000e-02  4.840500e-02          low  1.453604                     0                    1                  False                 False)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables
# metric_df, thresholds_metric, metric_earliest_anoms, earliest_anoms_top20 expected to exist
try:
    metric_df
    thresholds_metric
    metric_earliest_anoms
    earliest_anoms_top20
except NameError as e:
    raise NameError(f"Required kernel variable missing: {e}")

# Settings
tz_sh = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
end_local   = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)
start_utc_ts = int(start_local.tz_convert(pytz.UTC).timestamp())
end_utc_ts   = int(end_local.tz_convert(pytz.UTC).timestamp())

# Target components
targets = ['Redis01', 'Redis02']

# 1) Identify metric groups for each Redis node that appeared in metric_earliest_anoms or earliest_anoms_top20
groups_from_metric_earliest = metric_earliest_anoms[['cmdb_id','kpi_name']].drop_duplicates()
groups_from_earliest_top20 = earliest_anoms_top20[earliest_anoms_top20['file_type']=='metric'][['cmdb_id','kpi_name']].drop_duplicates()

groups_union = pd.concat([groups_from_metric_earliest, groups_from_earliest_top20], ignore_index=True).drop_duplicates()
# Restrict to Redis01/Redis02
redis_groups = groups_union[groups_union['cmdb_id'].isin(targets)].reset_index(drop=True)

# Ensure metric_df numeric types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Helper to compute consecutive runs and longest run info
def longest_consec_info(timestamps, ratios):
    # timestamps: numpy array of integer seconds sorted ascending
    # ratios: numpy array aligned with timestamps
    if len(timestamps) == 0:
        return 0, False, np.nan  # length, consec_bool, max_ratio
    ts = np.sort(timestamps.astype(int))
    # Identify runs where consecutive diffs == 60
    diffs = np.diff(ts)
    runs = []
    run_start_idx = 0
    run_len = 1
    for i, d in enumerate(diffs):
        if d == 60:
            run_len += 1
        else:
            runs.append((run_start_idx, run_start_idx + run_len - 1, run_len))
            run_start_idx = i+1
            run_len = 1
    runs.append((run_start_idx, run_start_idx + run_len - 1, run_len))
    # Find longest run length
    longest = max(r[2] for r in runs) if runs else 1
    consec_bool = longest >= 2
    # Compute max ratio among runs that have length == longest
    max_ratio = np.nan
    for start, end, length in runs:
        if length == longest:
            # select ratios among indices start..end
            # map start/end to original indices after sorting
            # Need to align ratios to sorted timestamps: create mapping
            # We'll sort timestamps and ratios together
            sorted_idx = np.argsort(timestamps.astype(int))
            selected = ratios[sorted_idx][start:end+1]
            if selected.size > 0:
                rmax = np.nanmax(selected)
                if np.isnan(max_ratio) or (rmax > max_ratio):
                    max_ratio = rmax
    return int(longest), bool(consec_bool), float(max_ratio) if not np.isnan(max_ratio) else np.nan

records = []

# Iterate each redis group
for _, row in redis_groups.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    # lookup global thresholds (must use precomputed thresholds_metric)
    thr = thresholds_metric[(thresholds_metric['cmdb_id']==cmdb) & (thresholds_metric['kpi_name']==kpi)]
    if thr.empty:
        continue
    p95 = float(thr['p95'].iloc[0])
    p05 = float(thr['p05'].iloc[0])
    # filter metric_df for group and incident window
    grp = metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==kpi) &
                    (metric_df['timestamp'] >= start_utc_ts) & (metric_df['timestamp'] <= end_utc_ts)].copy()
    if grp.empty:
        continue
    grp = grp.dropna(subset=['value']).sort_values('timestamp').reset_index(drop=True)
    total_points = int(grp.shape[0])
    # masks
    high_mask = (grp['value'] > p95) if not np.isnan(p95) else pd.Series(False, index=grp.index)
    low_mask  = (grp['value'] < p05) if not np.isnan(p05) else pd.Series(False, index=grp.index)
    high_count = int(high_mask.sum())
    low_count  = int(low_mask.sum())
    if high_count==0 and low_count==0:
        continue  # skip groups with no breach in-window
    # earliest anomaly timestamp (prefer high if equal)
    earliest_high_ts = int(grp.loc[high_mask, 'timestamp'].min()) if high_count>0 else None
    earliest_low_ts  = int(grp.loc[low_mask, 'timestamp'].min())  if low_count>0  else None
    if (earliest_high_ts is not None) and (earliest_low_ts is not None):
        if earliest_high_ts <= earliest_low_ts:
            earliest_anom_ts = earliest_high_ts
        else:
            earliest_anom_ts = earliest_low_ts
    elif earliest_high_ts is not None:
        earliest_anom_ts = earliest_high_ts
    else:
        earliest_anom_ts = earliest_low_ts
    earliest_anom_dt = pd.to_datetime(int(earliest_anom_ts), unit='s', utc=True).tz_convert(tz_sh)
    # For high anomalies: compute timestamps and ratios
    if high_count>0:
        high_ts = grp.loc[high_mask, 'timestamp'].values.astype('int64')
        high_vals = grp.loc[high_mask, 'value'].values.astype(float)
        # compute ratios value/p95, handle p95==0 -> inf
        if p95 == 0:
            high_ratios = np.array([np.inf]*len(high_vals), dtype=float)
        else:
            high_ratios = high_vals / p95
        high_longest_len, high_consec_bool, high_max_ratio = longest_consec_info(high_ts, high_ratios)
    else:
        high_longest_len, high_consec_bool, high_max_ratio = 0, False, np.nan
    # For low anomalies: timestamps and ratios p05/value
    if low_count>0:
        low_ts = grp.loc[low_mask, 'timestamp'].values.astype('int64')
        low_vals = grp.loc[low_mask, 'value'].values.astype(float)
        # compute ratios p05/value, handle value==0 -> inf
        low_ratios = np.array([ (np.inf if v==0 else (p05/ v)) for v in low_vals ], dtype=float)
        low_longest_len, low_consec_bool, low_max_ratio = longest_consec_info(low_ts, low_ratios)
    else:
        low_longest_len, low_consec_bool, low_max_ratio = 0, False, np.nan
    # Decide overall longest consecutive sequence (choose longer; if tie choose higher max_ratio)
    if high_longest_len > low_longest_len:
        longest_len = high_longest_len
        consecutive_bool = high_consec_bool
        max_ratio = high_max_ratio
    elif low_longest_len > high_longest_len:
        longest_len = low_longest_len
        consecutive_bool = low_consec_bool
        max_ratio = low_max_ratio
    else:
        # tie: pick the larger ratio (handle NaN)
        if np.isnan(high_max_ratio) and np.isnan(low_max_ratio):
            longest_len = high_longest_len
            consecutive_bool = high_consec_bool or low_consec_bool
            max_ratio = np.nan
        elif np.isnan(high_max_ratio):
            longest_len = low_longest_len
            consecutive_bool = low_consec_bool
            max_ratio = low_max_ratio
        elif np.isnan(low_max_ratio):
            longest_len = high_longest_len
            consecutive_bool = high_consec_bool
            max_ratio = high_max_ratio
        else:
            # both numeric: choose higher ratio
            if high_max_ratio >= low_max_ratio:
                longest_len = high_longest_len
                consecutive_bool = high_consec_bool
                max_ratio = high_max_ratio
            else:
                longest_len = low_longest_len
                consecutive_bool = low_consec_bool
                max_ratio = low_max_ratio
    records.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'total_points_in_window': total_points,
        'high_count': high_count,
        'low_count': low_count,
        'earliest_anomaly_timestamp': earliest_anom_dt,
        'consecutive_bool': bool(consecutive_bool),
        'longest_consec_len': int(longest_len),
        'max_ratio': float(max_ratio) if not np.isnan(max_ratio) else np.nan,
        # keep detailed per-type info for full storage
        'high_longest_len': int(high_longest_len),
        'low_longest_len': int(low_longest_len),
        'high_max_ratio': float(high_max_ratio) if not np.isnan(high_max_ratio) else np.nan,
        'low_max_ratio': float(low_max_ratio) if not np.isnan(low_max_ratio) else np.nan
    })

# Build DataFrame and sort
redis_consec_summary = pd.DataFrame.from_records(records)
if not redis_consec_summary.empty:
    redis_consec_summary = redis_consec_summary.sort_values(by=['earliest_anomaly_timestamp','max_ratio'], ascending=[True, False]).reset_index(drop=True)

# Prepare compact display (limit to top 30 rows), select requested columns
display_cols = ['cmdb_id','kpi_name','total_points_in_window','high_count','low_count',
                'earliest_anomaly_timestamp','consecutive_bool','longest_consec_len','max_ratio']
if not redis_consec_summary.empty:
    redis_summary_display = redis_consec_summary[display_cols].head(30).copy()
    # round max_ratio for compactness, keep inf
    redis_summary_display['max_ratio'] = redis_summary_display['max_ratio'].apply(
        lambda x: (float('inf') if np.isinf(x) else (np.nan if pd.isna(x) else round(float(x),6)))
    )
else:
    redis_summary_display = pd.DataFrame(columns=display_cols)

# Output only the compact display; full per-group sequence info is stored in redis_consec_summary
redis_summary_display

```
Out[7]:
```
```python
# Summarize Redis01 and Redis02 consecutive-anomaly findings in plain English (compact)
try:
    redis_summary_display
    redis_consec_summary
except NameError:
    summary = "Redis consecutive-summary results not found in kernel."
else:
    # basic counts
    n_rows = int(redis_summary_display.shape[0])
    # identify top concerns by max_ratio
    top_by_ratio = redis_summary_display.sort_values(['max_ratio'], ascending=False).head(5)
    top_items = []
    for _, r in top_by_ratio.iterrows():
        top_items.append(f"{r['cmdb_id']} | {r['kpi_name']} | earliest={r['earliest_anomaly_timestamp'].strftime('%Y-%m-%d %H:%M:%S%z')} | max_ratio={r['max_ratio']}")
    # count consecutive groups
    consec_count = int(redis_summary_display['consecutive_bool'].sum())
    # earliest anomaly timestamp overall
    earliest_ts = redis_summary_display['earliest_anomaly_timestamp'].min().strftime('%Y-%m-%d %H:%M:%S%z') if n_rows>0 else 'N/A'
    # Compose summary
    lines = [
        f"Redis consecutive-anomaly summary (window 2021-03-06 02:00–02:30 UTC+8):",
        f"- Groups reported: {n_rows}. Earliest anomaly observed at {earliest_ts}.",
        f"- Consecutive anomaly sequences (length>=2) detected in {consec_count} groups (notably many redis-related KPIs).",
        "- Top concerns by max_ratio:",
    ]
    lines += [f"  • {s}" for s in top_items]
    lines += [
        "- Key observations:",
        "  1) Highest severity: Redis02 OSLinux-CPU_CPU_CPUWio (ratio ~1.6099) — investigate host CPU WIO on Redis02 first.",
        "  2) Next: Redis01 OSLinux-CPU_CPU_CPUWio (ratio ~1.2191). Both indicate elevated CPU I/O wait around 02:00.",
        "  3) Many Redis metrics (used_memory_rss, total_connections, uptime, keyspace_hits, rdb_changes, etc.) show low anomalies starting at 02:00 and form short consecutive runs (longest length = 2). These suggest a system-wide Redis behavior change at that time.",
        "  4) Most ratios are very close to 1.0 (small deviations) except the CPU WIO entries — so prioritize CPU/IO issues first, then investigate Redis application-level metrics.",
        "",
        "- Recommended next steps:",
        "  1) Check Redis02 host disk/CPU I/O and kernel I/O wait metrics around 02:00.",
        "  2) On Redis01/Redis02, review Redis logs and client connection patterns around 02:00 (total_connections, keyspace_hits).",
        "  3) Correlate with system-level I/O and network metrics and with any error logs around those timestamps.",
    ]
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  high_count  low_count earliest_anomaly_timestamp  consecutive_bool  longest_consec_len  max_ratio
0   Redis02                             OSLinux-CPU_CPU_CPUWio                      31           2          0  2021-03-06 02:00:00+08:00             False                   1   1.609895
1   Redis01                             OSLinux-CPU_CPU_CPUWio                      29           2          1  2021-03-06 02:00:00+08:00             False                   1   1.219078
2   Redis02        OSLinux-NTP_197.30.1.67_NtpServerTimeOffset                       5           1          1  2021-03-06 02:00:00+08:00             False                   1   1.142857
3   Redis01          redis-Redis_6379_Redis  (used_memory_rss)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.005308
4   Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...                       7           0          1  2021-03-06 02:00:00+08:00             False                   1   1.001852
5   Redis01  redis-Redis_6379_Redis  (total_connections_rec...                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000020
6   Redis01                redis-Redis_6379_Redis  (lru_clock)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000020
7   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime                      31           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000020
8   Redis01        redis-Redis_6379_Redis  (uptime_in_seconds)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000019
9   Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime                      29           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000014
10  Redis01             redis-Redis_6379_Redis  (used_cpu_sys)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000004
11  Redis01            redis-Redis_6379_Redis  (used_cpu_user)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000004
12  Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000002
13  Redis01  redis-Redis_6379_Redis  (total_commands_proces...                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000002
14  Redis02  redis-Redis_6379_Redis  (rdb_changes_since_las...                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000002
15  Redis01            redis-Redis_6379_Redis  (keyspace_hits)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000002
16  Redis02            redis-Redis_6379_Redis  (keyspace_hits)                      30           2          2  2021-03-06 02:00:00+08:00              True                   2   1.000002```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables
try:
    metric_df
    thresholds_metric
except NameError as e:
    raise NameError("Required kernel variables missing: " + str(e))

# Settings
tz_sh = pytz.timezone('Asia/Shanghai')
start_local = pd.Timestamp('2021-03-06 02:00:00', tz=tz_sh)
end_local   = pd.Timestamp('2021-03-06 02:30:00', tz=tz_sh)
start_utc_ts = int(start_local.tz_convert(pytz.UTC).timestamp())
end_utc_ts   = int(end_local.tz_convert(pytz.UTC).timestamp())

cmdb = 'Redis02'
exact_kpi = 'OSLinux-CPU_CPU_CPUWio'
prefix_kpi = 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda'

# Ensure numeric columns
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Identify local-disk KPIs for Redis02 with given prefix that have any points in the full metric_df
candidate_disk_kpis = metric_df[
    (metric_df['cmdb_id'] == cmdb) &
    (metric_df['kpi_name'].str.startswith(prefix_kpi))
]['kpi_name'].unique().tolist()

# Build final KPI list: include exact_kpi if present and any disk KPIs with points
kpis = []
if metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==exact_kpi)].shape[0] > 0:
    kpis.append(exact_kpi)
kpis += [k for k in candidate_disk_kpis if pd.notna(k)]

# Prepare storage for full extracted per-kpi point tables
redis02_kpi_points = {}

# Helper to compute longest consecutive runs and max ratio during those runs
def compute_runs_and_max_ratio(ts_arr, ratio_arr):
    # ts_arr: numpy array of integer timestamps sorted ascending
    # ratio_arr: numpy array of floats aligned
    if len(ts_arr) == 0:
        return 0, False, np.nan
    ts = np.array(ts_arr, dtype='int64')
    order = np.argsort(ts)
    ts = ts[order]
    ratios = np.array(ratio_arr, dtype=float)[order]
    # compute diffs
    diffs = np.diff(ts)
    runs = []
    run_start = 0
    run_len = 1
    for i, d in enumerate(diffs):
        if d == 60:
            run_len += 1
        else:
            runs.append((run_start, run_start + run_len - 1, run_len))
            run_start = i + 1
            run_len = 1
    runs.append((run_start, run_start + run_len - 1, run_len))
    # longest
    longest = max(r[2] for r in runs) if runs else 1
    consec_bool = longest >= 2
    # compute max ratio among runs with length == longest
    max_ratio = np.nan
    for start, end, length in runs:
        if length == longest:
            seg = ratios[start:end+1]
            if seg.size>0:
                seg_max = np.nanmax(seg)
                if np.isnan(max_ratio) or seg_max > max_ratio:
                    max_ratio = seg_max
    return int(longest), bool(consec_bool), (float(max_ratio) if not np.isnan(max_ratio) else np.nan)

# Collect display rows (limit 20 per KPI) and summary records
display_rows = []
summary_records = []

for kpi in kpis:
    # get global thresholds for this (cmdb,kpi)
    thr = thresholds_metric[(thresholds_metric['cmdb_id']==cmdb) & (thresholds_metric['kpi_name']==kpi)]
    if thr.empty:
        # No precomputed global thresholds -> skip (RULE 9)
        continue
    p95 = float(thr['p95'].iloc[0])
    p05 = float(thr['p05'].iloc[0])
    # filter raw metric rows for Redis02, this KPI, in-window
    grp = metric_df[(metric_df['cmdb_id']==cmdb) & (metric_df['kpi_name']==kpi) &
                    (metric_df['timestamp'] >= start_utc_ts) & (metric_df['timestamp'] <= end_utc_ts)].copy()
    if grp.empty:
        continue
    # annotate breach booleans
    grp = grp.sort_values('timestamp').reset_index(drop=True)
    grp['breach_high'] = np.where((~grp['value'].isna()) & (~np.isnan(p95)) & (grp['value'] > p95), True, False)
    grp['breach_low']  = np.where((~grp['value'].isna()) & (~np.isnan(p05)) & (grp['value'] < p05), True, False)
    # convert timestamp to tz-aware datetime for display
    grp['timestamp_dt'] = pd.to_datetime(grp['timestamp'].astype(int), unit='s', utc=True).dt.tz_convert(tz_sh)
    # store full table in redis02_kpi_points dict
    redis02_kpi_points[kpi] = grp[['timestamp','timestamp_dt','value','breach_high','breach_low']].copy()
    # prepare up-to-20 display rows
    disp = redis02_kpi_points[kpi].head(20).copy()
    disp.insert(0, 'kpi_name', kpi)
    display_rows.append(disp)
    # summary computations
    total_points = int(grp.shape[0])
    high_count = int(grp['breach_high'].sum())
    low_count  = int(grp['breach_low'].sum())
    # earliest breach timestamp (first where either breach_high or breach_low)
    mask_any = grp['breach_high'] | grp['breach_low']
    if mask_any.sum() == 0:
        continue  # skip if no breach in-window (shouldn't happen since we filtered earlier but safe)
    earliest_idx = grp.index[mask_any][0]
    earliest_ts = int(grp.loc[earliest_idx,'timestamp'])
    earliest_ts_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz_sh)
    # For runs: we need timestamps and ratios for points where breach_any True
    breach_rows = grp[mask_any].copy()
    # compute per-point ratio: for high use value/p95 (p95==0 -> inf), for low use p05/value (val==0->inf)
    def point_ratio(r):
        v = float(r['value'])
        if r['breach_high']:
            return (np.inf if p95==0 else (v / p95))
        else:
            return (np.inf if v==0 else (p05 / v))
    breach_rows['ratio'] = breach_rows.apply(point_ratio, axis=1)
    ts_arr = breach_rows['timestamp'].values.astype('int64')
    ratio_arr = breach_rows['ratio'].values.astype(float)
    longest_len, consec_bool, max_ratio = compute_runs_and_max_ratio(ts_arr, ratio_arr)
    summary_records.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'total_points_in_window': total_points,
        'high_count': high_count,
        'low_count': low_count,
        'earliest_breach_timestamp': earliest_ts_dt,
        'longest_consecutive_breach_length': longest_len,
        'consecutive_bool': consec_bool,
        'max_ratio_during_longest_consec': max_ratio
    })

# Prepare compact display: concatenate per-kpi up-to-20 rows
if display_rows:
    redis02_points_display = pd.concat(display_rows, ignore_index=True)
    # select and rename columns for compactness
    redis02_points_display = redis02_points_display[['kpi_name','timestamp_dt','value','breach_high','breach_low']]
    redis02_points_display = redis02_points_display.rename(columns={'timestamp_dt':'timestamp'})
else:
    redis02_points_display = pd.DataFrame(columns=['kpi_name','timestamp','value','breach_high','breach_low'])

# Build summary DataFrame
redis02_summary_df = pd.DataFrame.from_records(summary_records)
if not redis02_summary_df.empty:
    # round max_ratio for compactness, keep inf as-is
    redis02_summary_df['max_ratio_during_longest_consec'] = redis02_summary_df['max_ratio_during_longest_consec'].apply(
        lambda x: (float('inf') if np.isinf(x) else (np.nan if pd.isna(x) else round(float(x),6)))
    )
    # sort by earliest breach ascending
    redis02_summary_df = redis02_summary_df.sort_values(by='earliest_breach_timestamp').reset_index(drop=True)

# Store full per-kpi point tables in kernel variable redis02_kpi_points (dict)
# Also store full summary in redis02_consec_kpi_summary_full
redis02_consec_kpi_summary_full = redis02_summary_df.copy()

# Final compact outputs: redis02_points_display (up to 20 rows per KPI) and redis02_summary_df (one row per KPI)
redis02_points_display, redis02_summary_df

```
Out[8]:
```
Summary for Redis02 (window 2021-03-06 02:00–02:30 UTC+8):

- Data extracted and stored: full in-window point tables for each KPI are saved in kernel variable redis02_kpi_points (keyed by KPI name). The displayed samples show up to 20 rows per KPI (138 rows total in the combined display).

- KPIs analyzed:
  1. OSLinux-CPU_CPU_CPUWio
     - total points: 31
     - high_count: 2, low_count: 0
     - earliest breach: 2021-03-06 02:00:00+08:00
     - longest consecutive breach length: 1 (no multi-minute consecutive run)
     - max ratio during longest consecutive breach: ~1.6099
     - Notes: high CPU I/O-wait spikes at ~02:00 (highest severity among Redis02 KPIs).
  2. OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg... (disk average latency)
     - total points: 7
     - high_count: 0, low_count: 1
     - earliest breach: 2021-03-06 02:00:00+08:00
     - longest_consec_len: 1
     - max_ratio: ~1.001852
  3. OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps (write TPS)
     - total points: 15
     - high_count: 1, low_count: 1
     - earliest breach: 2021-03-06 02:03:00+08:00
     - longest_consec_len: 1
     - max_ratio: ~1.072366
  4. OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite (write KB/s)
     - total points: 15
     - high_count: 1, low_count: 1
     - earliest breach: 2021-03-06 02:03:00+08:00
     - longest_consec_len: 1
     - max_ratio: ~1.616816
     - Notes: notable write-rate spike (ratio ~1.617).
  5. OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead...
     - total points: 10
     - high_count: 1, low_count: 1
     - earliest breach: 2021-03-06 02:04:00+08:00
     - longest_consec_len: 1
     - max_ratio: ~1.416894
  6. OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...
     - total points: 15
     - high_count: 1, low_count: 0
     - earliest breach: 2021-03-06 02:09:00+08:00
     - longest_consec_len: 1
     - max_ratio: ~3.333333

- Overall observations and priority:
  - The highest-severity signal for Redis02 is the CPU I/O-wait KPI (OSLinux-CPU_CPU_CPUWio) with max ratio ~1.61 at 02:00 — investigate host CPU I/O wait and underlying disk I/O around 02:00 first.
  - Disk write metric (sda_DSKWrite) also has a notable spike (ratio ~1.62) at ~02:03 — inspect disk write activity and I/O queueing.
  - Most other disk-related breaches are single-minute and have small ratios (~1.00–1.42), but one metric shows ratio ~3.33 at 02:09 (needs review).
  - Consecutive multi-minute runs are uncommon; several Redis-related metrics on Redis02 show only single-minute breaches (longest consecutive = 1), so issues appear as short spikes rather than prolonged multi-minute degradation—except check the points with consecutive=True in the stored tables if needed.

- Next steps:
  1. Investigate host-level I/O wait (CPU WIO) on Redis02 at ~02:00.
  2. Correlate with disk write activity (sda_DSKWrite at ~02:03) and the DSKPer spike at ~02:09.
  3. Review Redis logs and client activity around these timestamps and check system-level I/O metrics (iostat, dmesg, storage backend metrics).

If you want, I can (a) print the stored per-KPI point table for any specific KPI (redis02_kpi_points['<KPI>']).head(20), or (b) return the full redis02_consec_kpi_summary_full table.

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name                 timestamp     value  breach_high  breach_low
0                              OSLinux-CPU_CPU_CPUWio 2021-03-06 02:00:00+08:00  0.436400         True       False
1                              OSLinux-CPU_CPU_CPUWio 2021-03-06 02:01:00+08:00  0.004200        False       False
2                              OSLinux-CPU_CPU_CPUWio 2021-03-06 02:02:00+08:00  0.565395         True       False
3                              OSLinux-CPU_CPU_CPUWio 2021-03-06 02:03:00+08:00  0.008300        False       False
4                              OSLinux-CPU_CPU_CPUWio 2021-03-06 02:04:00+08:00  0.004200        False       False
..                                                ...                       ...       ...          ...         ...
133  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite 2021-03-06 02:21:00+08:00  8.000000        False       False
134  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite 2021-03-06 02:23:00+08:00  7.733300        False       False
135  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite 2021-03-06 02:25:00+08:00  8.400000        False       False
136  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite 2021-03-06 02:27:00+08:00  7.866700        False       False
137  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite 2021-03-06 02:29:00+08:00  8.933333        False       False

[138 rows x 5 columns],    cmdb_id                                           kpi_name  total_points_in_window  high_count  low_count earliest_breach_timestamp  longest_consecutive_breach_length  consecutive_bool  max_ratio_during_longest_consec
0  Redis02                             OSLinux-CPU_CPU_CPUWio                      31           2          0 2021-03-06 02:00:00+08:00                                  1             False                         1.609895
1  Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...                       7           0          1 2021-03-06 02:00:00+08:00                                  1             False                         1.001852
2  Redis02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps                      15           1          1 2021-03-06 02:03:00+08:00                                  1             False                         1.072366
3  Redis02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite                      15           1          1 2021-03-06 02:03:00+08:00                                  1             False                         1.616816
4  Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                      10           1          1 2021-03-06 02:04:00+08:00                                  1             False                         1.416894
5  Redis02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...                      15           1          0 2021-03-06 02:09:00+08:00                                  1             False                         3.333333)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```